# 07 — Final frozen-detector steganalysis on the test split

This notebook is the **post-freeze detectability evaluation**. It does not tune the encoder, the payloads, the SRM-derived local-risk model, the allocator weight, or the detector.

The independent detector is the enhanced residual CNN frozen in **05e**. It was trained only on the training split and used on validation to confirm transfer before the allocator was frozen. The 2000-image test split did not participate in detector fitting or allocator selection.

### Confirmatory endpoint

The pre-specified primary contrast is:

\[
\alpha=0.25\ \text{joint allocator}
\quad \text{vs.}\quad
\alpha=1.0\ \text{predictability-only allocator}
\]

at the already frozen teacher/diagnostic payload:

\[
R=0.009\ \text{net bpp}.
\]

The primary endpoint is the paired difference in enhanced-CNN score change:

\[
\big[s(Y_{\rm joint})-s(X)\big]-
\big[s(Y_{\rm pred})-s(X)\big].
\]

A negative difference with a two-sided 95% bootstrap CI entirely below zero confirms lower detector response for the frozen joint allocator on the held-out test set.

All other payload/strategy contrasts are secondary. **No result in this notebook may be used to retune the method.**


In [ ]:
from pathlib import Path
import json, joblib, yaml, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.pipeline import prepare_image_context, run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.detector_repair import load_enhanced_cnn, score_enhanced_residual_cnn
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.final_steganalysis import validate_test_completion, primary_endpoint_decision

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
bs=int(config['dataset']['block_size'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

manifest_path=Path(config['dataset']['prepared_manifest'])
manifest=pd.read_csv(manifest_path)
test=manifest[manifest.split=='test'].reset_index(drop=True)
assert len(test)==2000
assert test.source_id.astype(str).is_unique

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
assert np.isclose(alpha,0.25)
payloads=list(map(float,allocator['payload_levels']))
strategies=list(config['allocator']['strategies'])
primary_bpp=float(allocator['teacher_payload_bpp'])
assert any(np.isclose(primary_bpp,p) for p in payloads)

out06=Path('/workspace/results/frozen_test_final')
complete=json.loads((out06/'test_run_complete.json').read_text())
validate_test_completion(complete, expected_cases=len(test)*len(payloads)*len(strategies))
protocol06=json.loads((out06/'test_protocol.json').read_text())
common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)

# Fail closed on frozen provenance.
cnn_path=Path('/workspace/results/models/enhanced_residual_cnn_05e.pt')
risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
prov=allocator['provenance_sha256']
if sha256_file(cnn_path) != prov['enhanced_residual_cnn_05e_pt']:
    raise RuntimeError('Frozen enhanced-CNN hash mismatch.')
if sha256_file(risk_path) != prov['srm_teacher_local_risk_joblib']:
    raise RuntimeError('Frozen local-risk model hash mismatch.')
if stable_id_hash(test.source_id.astype(str).tolist()) != complete['test_source_ids_sha256']:
    raise RuntimeError('Frozen test source-ID hash mismatch.')

context_dir=out06/'contexts'/protocol06['context_tag']
if not context_dir.exists():
    raise RuntimeError(f'Frozen test context cache missing: {context_dir}')

local_risk=joblib.load(risk_path)
cnn,device=load_enhanced_cnn(cnn_path)

out=Path('/workspace/results/final_steganalysis')
score_dir=out/'score_checkpoints'
score_dir.mkdir(parents=True,exist_ok=True)

print('Frozen alpha:',alpha)
print('Payloads:',payloads)
print('Strategies:',strategies)
print('Primary payload:',primary_bpp)
print('CNN device:',device)
print('06 common-feasible counts:')
print(common.groupby('target_net_bpp').size())
print('No retuning is permitted.')


In [ ]:
# Load test covers once and score them once with the frozen detector.
cover_images=[read_gray(p) for p in test.path]
test_ids=test.source_id.astype(str).tolist()
id_to_index={sid:i for i,sid in enumerate(test_ids)}

cover_scores=score_enhanced_residual_cnn(
    cnn,cover_images,device=device,batch_size=16
)
cover_score_map=dict(zip(test_ids,map(float,cover_scores)))

pd.DataFrame({
    'source_id':test_ids,
    'cover_score':cover_scores,
}).to_csv(out/'cover_scores.csv',index=False)

print('Scored',len(cover_scores),'test covers')
print('cover score mean/std:',float(np.mean(cover_scores)),float(np.std(cover_scores)))


In [ ]:
def _bpp_tag(bpp):
    return f'{float(bpp):.3f}'.replace('.','p')

def _load_context(x,sid):
    p=context_dir/f'{sid}.joblib'
    if p.exists():
        ctx=joblib.load(p)
        if ctx.get('context_tag')!=protocol06['context_tag']:
            raise RuntimeError(f'Stale context for {sid}')
        return ctx['orders'],ctx['block_rows'],ctx['plans']
    # Fallback should normally not be needed after complete 06.
    return prepare_image_context(x,sid,local_risk,alpha,bs,seed)

def _expected_ids_for_payload(bpp):
    ids=common[np.isclose(common.target_net_bpp.astype(float),float(bpp))].source_id.astype(str).tolist()
    if len(ids)==0:
        raise RuntimeError(f'No common-feasible IDs for payload {bpp}')
    return ids

def score_strategy_payload(strategy,bpp,batch_size=16):
    ids=_expected_ids_for_payload(bpp)
    ck=score_dir/f'{strategy}_bpp_{_bpp_tag(bpp)}.csv'

    # Combination-level resume.
    if ck.exists():
        old=pd.read_csv(ck)
        old['source_id']=old.source_id.astype(str)
        if (
            len(old)==len(ids)
            and old.source_id.tolist()==ids
            and np.allclose(old.target_net_bpp.astype(float),float(bpp))
            and set(old.strategy.astype(str))=={strategy}
        ):
            print('resume:',strategy,bpp,'rows',len(old))
            return old
        raise RuntimeError(f'Existing score checkpoint is inconsistent: {ck}')

    rows=[]
    batch_imgs=[]
    batch_meta=[]

    def flush():
        nonlocal batch_imgs,batch_meta,rows
        if not batch_imgs:
            return
        ss=score_enhanced_residual_cnn(cnn,batch_imgs,device=device,batch_size=batch_size)
        for meta,score in zip(batch_meta,ss):
            sid=meta['source_id']
            cs=cover_score_map[sid]
            rows.append({
                **meta,
                'cover_score':float(cs),
                'stego_score':float(score),
                'score_delta':float(score-cs),
            })
        batch_imgs=[]; batch_meta=[]

    for k,sid in enumerate(ids,1):
        i=id_to_index[sid]
        x=cover_images[i]
        orders,br,plans=_load_context(x,sid)
        rr=run_frozen_image_precomputed(
            x,sid,float(bpp),strategy,orders,br,bs,seed,
            False,None,plans=plans
        )
        if not rr['feasible']:
            raise RuntimeError(f'Case was common-feasible in 06 but is infeasible now: {sid} {strategy} {bpp}')
        if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
            raise RuntimeError(f'Reversibility invariant failed while regenerating {sid} {strategy} {bpp}')

        # Cross-check against the frozen 06 record.
        old=per06[
            (per06.source_id==sid) &
            (per06.strategy.astype(str)==strategy) &
            np.isclose(per06.target_net_bpp.astype(float),float(bpp))
        ]
        if len(old)!=1:
            raise RuntimeError(f'Could not find unique 06 case for {sid} {strategy} {bpp}')
        old=old.iloc[0]
        if int(rr['net_payload_bits']) != int(old['net_payload_bits']):
            raise RuntimeError(f'Net-payload mismatch vs 06 for {sid} {strategy} {bpp}')
        if not np.isclose(float(rr['psnr']),float(old['psnr']),rtol=0,atol=1e-10):
            raise RuntimeError(f'PSNR mismatch vs 06 for {sid} {strategy} {bpp}')

        batch_imgs.append(rr['stego'])
        batch_meta.append({
            'source_id':sid,
            'strategy':strategy,
            'target_net_bpp':float(bpp),
        })
        if len(batch_imgs)>=batch_size:
            flush()
        if k%250==0 or k==len(ids):
            print(strategy,bpp,k,'/',len(ids))

    flush()
    df=pd.DataFrame(rows)
    if len(df)!=len(ids) or df.source_id.astype(str).tolist()!=ids:
        raise RuntimeError('Score output alignment failure.')
    df.to_csv(ck,index=False)
    return df

frames=[]
for bpp in payloads:
    for strategy in strategies:
        frames.append(score_strategy_payload(strategy,bpp))

scores=pd.concat(frames,ignore_index=True)
scores.to_csv(out/'enhanced_cnn_test_scores.csv',index=False)
print('Saved detector scores:',len(scores))


In [ ]:
# Detector metrics for each frozen strategy/payload on the SAME common-feasible covers.
summary_rows=[]
for (strategy,bpp),g in scores.groupby(['strategy','target_net_bpp'],sort=False):
    c=g.cover_score.to_numpy(float)
    s=g.stego_score.to_numpy(float)
    y=np.tile([0,1],len(g))
    sc=np.column_stack([c,s]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(
        c,s,fixed_fpr=fixed_fpr,
        n_resamples=n_boot,confidence=confidence,
        seed=seed+int(round(float(bpp)*1_000_000))+strategies.index(strategy)*1009,
    )
    summary_rows.append({
        'strategy':strategy,
        'target_net_bpp':float(bpp),
        'pairs':len(g),
        'auc':float(m['auc']),
        'auc_ci_low':float(ci['auc_low']),
        'auc_ci_high':float(ci['auc_high']),
        'tpr_at_5pct_fpr':float(m['tpr_at_fpr']),
        'tpr_ci_low':float(ci['tpr_low']),
        'tpr_ci_high':float(ci['tpr_high']),
        'score_delta_mean':float(np.mean(s-c)),
        'score_delta_median':float(np.median(s-c)),
        'cover_score_mean':float(np.mean(c)),
        'stego_score_mean':float(np.mean(s)),
    })

summary=pd.DataFrame(summary_rows).sort_values(['target_net_bpp','strategy']).reset_index(drop=True)
summary.to_csv(out/'enhanced_cnn_test_summary.csv',index=False)
display(summary)


In [ ]:
# Paired method comparisons: frozen JOINT vs each baseline, aligned by source image.
comparison_rows=[]
references=[s for s in strategies if s!='joint']

for bpp in payloads:
    ids=_expected_ids_for_payload(bpp)
    joint=(scores[
        (scores.strategy=='joint') &
        np.isclose(scores.target_net_bpp.astype(float),float(bpp))
    ].set_index('source_id').loc[ids])
    c=joint.cover_score.to_numpy(float)
    a=joint.stego_score.to_numpy(float)

    for ref in references:
        rr=(scores[
            (scores.strategy==ref) &
            np.isclose(scores.target_net_bpp.astype(float),float(bpp))
        ].set_index('source_id').loc[ids])
        if not np.allclose(c,rr.cover_score.to_numpy(float),rtol=0,atol=0):
            raise RuntimeError(f'Cover-score alignment failure: {bpp} {ref}')
        comp=paired_method_bootstrap(
            c,a,rr.stego_score.to_numpy(float),
            fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,
            seed=seed+70000+int(round(float(bpp)*1_000_000))+references.index(ref)*997,
        )
        comp.update({
            'method':'joint',
            'reference':ref,
            'target_net_bpp':float(bpp),
        })
        comparison_rows.append(comp)

comparisons=pd.DataFrame(comparison_rows)
comparisons.to_csv(out/'paired_joint_vs_baselines.csv',index=False)
display(comparisons)


In [ ]:
# Locked primary endpoint: joint vs predictability at 0.009 net bpp.
primary=comparisons[
    np.isclose(comparisons.target_net_bpp.astype(float),primary_bpp) &
    (comparisons.reference=='predictability')
]
if len(primary)!=1:
    raise RuntimeError('Primary comparison was not uniquely identified.')
primary=primary.iloc[0].to_dict()

decision=primary_endpoint_decision(primary)
decision.update({
    'primary_payload_bpp':primary_bpp,
    'method':'joint',
    'reference':'predictability',
    'alpha':alpha,
    'pairs':int(primary['n_pairs']),
    'primary_endpoint':'paired frozen enhanced-CNN score-change difference: joint minus predictability',
    'auc_joint':float(primary['auc_method']),
    'auc_predictability':float(primary['auc_reference']),
    'auc_diff':float(primary['auc_diff']),
    'auc_diff_ci_low':float(primary['auc_diff_low']),
    'auc_diff_ci_high':float(primary['auc_diff_high']),
    'tpr_joint':float(primary['tpr_method']),
    'tpr_predictability':float(primary['tpr_reference']),
    'tpr_diff':float(primary['tpr_diff']),
    'tpr_diff_ci_low':float(primary['tpr_diff_low']),
    'tpr_diff_ci_high':float(primary['tpr_diff_high']),
    'fixed_fpr':fixed_fpr,
    'confidence':confidence,
    'bootstrap_resamples':n_boot,
    'test_split_used_for_retuning':False,
    'no_retuning_permitted':True,
    'detector_sha256':sha256_file(cnn_path),
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
})
(out/'primary_endpoint.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
print(json.dumps(decision,indent=2))

if decision['primary_confirmed']:
    print('\nPRIMARY TEST ENDPOINT CONFIRMED.')
    print('The frozen joint allocator produced a lower independent-CNN score change than predictability-only at 0.009 net bpp.')
else:
    print('\nPRIMARY TEST ENDPOINT NOT CONFIRMED.')
    print('Do not retune the allocator. Report the held-out result as observed.')


In [ ]:
# Publication-oriented figures.
fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in strategies:
    z=summary[summary.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,z.auc,marker='o',label=strategy)
ax.axhline(0.5,linewidth=1)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Enhanced-CNN ROC-AUC')
ax.set_title('Held-out test detectability')
ax.grid(True,alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(out/'test_auc_vs_payload.png',dpi=300)
plt.show()

fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in strategies:
    z=summary[summary.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,z.tpr_at_5pct_fpr,marker='o',label=strategy)
ax.axhline(fixed_fpr,linewidth=1)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('TPR at 5% FPR')
ax.set_title('Held-out test detection at fixed false-positive rate')
ax.grid(True,alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(out/'test_tpr5_vs_payload.png',dpi=300)
plt.show()

jp=comparisons[comparisons.reference=='predictability'].sort_values('target_net_bpp')
fig,ax=plt.subplots(figsize=(6.6,4.5))
y=jp.delta_mean_diff.to_numpy(float)
lo=jp.delta_mean_diff_low.to_numpy(float)
hi=jp.delta_mean_diff_high.to_numpy(float)
ax.errorbar(jp.target_net_bpp,y,yerr=np.vstack([y-lo,hi-y]),marker='o',capsize=4)
ax.axhline(0,linewidth=1)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Paired CNN score-change difference\njoint − predictability')
ax.set_title('Frozen joint allocator vs predictability-only')
ax.grid(True,alpha=.2)
fig.tight_layout()
fig.savefig(out/'joint_vs_predictability_score_difference.png',dpi=300)
plt.show()


## Interpretation boundary

The enhanced residual CNN is the confirmatory independent detector because it is not the SRM-derived teacher used to construct \(D_i\). The SRM teacher itself should not be presented as independent evidence.

The test result must be reported without changing \(\alpha\), payload levels, local-risk model, detector, inclusion rules, or strategy definitions afterward.

Next, notebook **08** should measure robustness and **end-to-end computational cost**, including local-risk/context construction. The `encode_ms` values from notebook 06 measure the embedding path after block context/order preparation and therefore should not be described as full allocation runtime.
